# NB11 — duplicate-aware spectral sensitivity

**Project:** `BANANA_RIPENESS_MULTISENSOR_Q1`

This notebook examines the concern that repeated exact spectral vectors may dominate training and external metrics.

## Scientific strategy

Two complementary analyses are performed.

### A. Frozen-prediction reweighting
The original NB04 predictions are kept unchanged and external metrics are recomputed so that each exact predictor vector contributes equal total weight.

### B. Duplicate-collapsed nested LOEO
For the **Spectral** modality, exact repeated `(experiment, spectral vector, stage)` observations are collapsed to one row. If the same spectral vector occurs with different stages, each distinct vector–stage association is retained; labels are never resolved by arbitrary majority vote.

The full nested LOEO selection is then repeated with the original compact model grid.

Four duplicate-aware representations are evaluated:
1. `RAW_DEDUP`
2. `SNV_DEDUP`
3. `DOMAIN_Z_ROW_DEDUP` — experiment-wise z-score using all unlabeled rows, matching NB10
4. `DOMAIN_Z_UNIQUE_DEDUP` — experiment-wise z-score using **unique unlabeled spectral vectors** only

The fourth arm checks whether NB10's DOMAIN_Z gain depended on duplicated readings influencing the target-batch mean and standard deviation.

## Guardrails

- Frozen NB04 and NB10 results are never overwritten.
- No outer labels are used to fit preprocessing.
- Exact label conflicts are audited and retained transparently.
- Cross-experiment exact spectral overlaps are reported explicitly.
- The external post-hoc winner is not used for model selection.

**Hardware:** CPU only.


In [ ]:
# 0. Mount Drive, dependencies and configuration
from google.colab import drive
drive.mount('/content/drive')

import sys, subprocess, importlib.util
if importlib.util.find_spec('mord') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mord==0.7'])
if importlib.util.find_spec('xgboost') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, shutil, time, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score, mean_absolute_error
)

import mord
import xgboost as xgb

warnings.filterwarnings('ignore')

SEED=2026
np.random.seed(SEED)

PROJECT=Path('/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1')
MASTER_PATH=PROJECT/'02_EXTRACTED_DATA'/'06_master_original_multisensor.csv'
PROTOCOL_PATH=PROJECT/'04_FROZEN_SPLITS'/'nested_loeo_protocol_v1.json'
RESULTS_ROOT=PROJECT/'06_RESULTS'

assert MASTER_PATH.exists(),MASTER_PATH
assert PROTOCOL_PATH.exists(),PROTOCOL_PATH
assert RESULTS_ROOT.exists(),RESULTS_ROOT

RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR=RESULTS_ROOT/f'NB11_DUPLICATE_AWARE_{RUN_ID}'
TABLE_DIR=RUN_DIR/'tables'
FIG_DIR=RUN_DIR/'figures'
TEXT_DIR=RUN_DIR/'text'
META_DIR=RUN_DIR/'metadata'
for d in [TABLE_DIR,FIG_DIR,TEXT_DIR,META_DIR]:
    d.mkdir(parents=True,exist_ok=True)

ALL_LABELS=[1,2,3,4,5]
OUTER_TESTS=[2,3,4,5]
MODELS=['OrdinalLogistic','PLSDA','SVM_RBF','RandomForest','XGBoost']
METHODS=[
    'RAW_DEDUP',
    'SNV_DEDUP',
    'DOMAIN_Z_ROW_DEDUP',
    'DOMAIN_Z_UNIQUE_DEDUP'
]
N_JOBS=max(1,min(8,(os.cpu_count() or 2)-1))

start_all=time.perf_counter()

print('Project:',PROJECT)
print('Output:',RUN_DIR)
print('CPU workers:',N_JOBS)
print('Methods:',METHODS)


In [ ]:
# 1. Frozen provenance, NB04 and NB10 pointers
def sha256_file(path,chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size),b''):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_MASTER_SHA='0a06fb3773f85dacee42e18eecf17e5e2a8e7f5e9508cd3d60bd4b66757e8ea8'
EXPECTED_PROTOCOL_SHA='86731ff9a292f1afd35a215105b14c4af3aa2bda5e8c4dad4df639d4d82ad930'

assert sha256_file(MASTER_PATH)==EXPECTED_MASTER_SHA
assert sha256_file(PROTOCOL_PATH)==EXPECTED_PROTOCOL_SHA

LATEST_NB04=RESULTS_ROOT/'LATEST_NB04_RUN.json'
LATEST_NB10=RESULTS_ROOT/'LATEST_NB10_RUN.json'
assert LATEST_NB04.exists(),LATEST_NB04
assert LATEST_NB10.exists(),LATEST_NB10

nb04_ptr=json.loads(LATEST_NB04.read_text(encoding='utf-8'))
nb10_ptr=json.loads(LATEST_NB10.read_text(encoding='utf-8'))

NB04_RUN=Path(nb04_ptr['run_dir'])
NB10_RUN=Path(nb10_ptr['run_dir'])

NB04_PRED=NB04_RUN/'predictions'/'outer_predictions_all_models.csv.gz'
NB04_METRICS=NB04_RUN/'tables'/'outer_loeo_metrics_all_models.csv'
NB04_RANK=NB04_RUN/'tables'/'inner_vs_outer_model_ranking.csv'
NB10_SELECTED=NB10_RUN/'tables'/'09_selected_model_performance_raw_snv_domainz.csv'

for p in [NB04_PRED,NB04_METRICS,NB04_RANK,NB10_SELECTED]:
    assert p.exists(),p
    print('Source:',p)

source_hashes=pd.DataFrame([
    {'artifact':str(MASTER_PATH.relative_to(PROJECT)),'sha256':sha256_file(MASTER_PATH),'bytes':MASTER_PATH.stat().st_size},
    {'artifact':str(PROTOCOL_PATH.relative_to(PROJECT)),'sha256':sha256_file(PROTOCOL_PATH),'bytes':PROTOCOL_PATH.stat().st_size},
    {'artifact':str(NB04_PRED.relative_to(PROJECT)),'sha256':sha256_file(NB04_PRED),'bytes':NB04_PRED.stat().st_size},
    {'artifact':str(NB04_METRICS.relative_to(PROJECT)),'sha256':sha256_file(NB04_METRICS),'bytes':NB04_METRICS.stat().st_size},
    {'artifact':str(NB04_RANK.relative_to(PROJECT)),'sha256':sha256_file(NB04_RANK),'bytes':NB04_RANK.stat().st_size},
    {'artifact':str(NB10_SELECTED.relative_to(PROJECT)),'sha256':sha256_file(NB10_SELECTED),'bytes':NB10_SELECTED.stat().st_size},
])
source_hashes.to_csv(META_DIR/'source_hashes.csv',index=False)

print('Frozen provenance: PASS')


In [ ]:
# 2. Load canonical original observations and define predictors
df=pd.read_csv(MASTER_PATH,low_memory=False).reset_index(drop=True)

df['test_id']=df['test_id'].astype(int)
df['stage']=df['stage'].astype(int)

assert len(df)==17662
assert df['master_id'].is_unique
assert sorted(df.test_id.unique().tolist())==OUTER_TESTS

TPH_FEATURES=['Temp-int','Press-int','Humid-int']
SPECTRAL_FEATURES=[
    'SpA410','SpB435','SpC460','SpD485','SpE510','SpF535',
    'SpG560','SpH585','SpR610','SpI645','SpS680','SpJ705',
    'SpT730','SpU760','SpV810','SpW860','SpK900','SpL940'
]
FULL_FEATURES=TPH_FEATURES+SPECTRAL_FEATURES

assert df[SPECTRAL_FEATURES+TPH_FEATURES].notna().all().all()

# Exact group identifiers. groupby.ngroup uses exact equality of parsed numeric values.
df['spectral_group_global']=df.groupby(
    SPECTRAL_FEATURES,sort=False,dropna=False
).ngroup()

df['full_group_global']=df.groupby(
    FULL_FEATURES,sort=False,dropna=False
).ngroup()

# Exact predictor + label groups within experiment.
df['spectral_label_group']=df.groupby(
    ['test_id']+SPECTRAL_FEATURES+['stage'],sort=False,dropna=False
).ngroup()

df['full_label_group']=df.groupby(
    ['test_id']+FULL_FEATURES+['stage'],sort=False,dropna=False
).ngroup()

print('Rows:',len(df))
print('Unique global spectral vectors:',df.spectral_group_global.nunique())
print('Unique global full TPH+Spectral vectors:',df.full_group_global.nunique())


In [ ]:
# 3. Duplicate and label-conflict audit
audit_rows=[]

for t,g in df.groupby('test_id',sort=True):
    for modality,group_col,label_group_col in [
        ('Spectral','spectral_group_global','spectral_label_group'),
        ('TPH_Spectral','full_group_global','full_label_group')
    ]:
        vector_stage_counts=g.groupby(group_col).stage.nunique()
        conflict_groups=vector_stage_counts[vector_stage_counts>1].index

        audit_rows.append({
            'test_id':int(t),
            'modality':modality,
            'n_rows':len(g),
            'n_unique_feature_vectors':int(g[group_col].nunique()),
            'feature_vector_duplicate_fraction':float(1-g[group_col].nunique()/len(g)),
            'n_unique_feature_label_combinations':int(g[label_group_col].nunique()),
            'feature_label_duplicate_fraction':float(1-g[label_group_col].nunique()/len(g)),
            'n_conflicting_feature_vectors':int(len(conflict_groups)),
            'rows_in_conflicting_vectors':int(g[g[group_col].isin(conflict_groups)].shape[0]),
        })

duplicate_audit=pd.DataFrame(audit_rows)
duplicate_audit.to_csv(TABLE_DIR/'01_duplicate_audit_by_experiment_modality.csv',index=False)
display(duplicate_audit.round(4))

# Cross-experiment exact spectral overlap.
cross=(df.groupby('spectral_group_global',as_index=False)
       .agg(n_rows=('master_id','size'),
            n_tests=('test_id','nunique'),
            n_stages=('stage','nunique'),
            tests=('test_id',lambda x:','.join(map(str,sorted(set(x))))),
            stages=('stage',lambda x:','.join(map(str,sorted(set(x)))))))

cross_shared=cross[cross.n_tests>1].copy()
cross_shared.to_csv(TABLE_DIR/'02_cross_experiment_exact_spectral_overlap.csv',index=False)

print('\nCross-experiment spectral groups:',len(cross_shared))
print('Rows in cross-experiment groups:',int(cross_shared.n_rows.sum()))
print('Shared groups with label conflict:',int((cross_shared.n_stages>1).sum()))
display(cross_shared)


## Frozen-prediction duplicate-aware reevaluation

The original NB04 models are not retrained in this section.

Two alternative external metrics are added:
- **feature-vector balanced:** every exact predictor vector receives total weight 1;
- **feature+label collapsed:** one row is retained per exact predictor-vector / true-stage combination.

The second approach preserves conflicting labels instead of forcing a majority class.


In [ ]:
# 4. Frozen NB04 prediction reweighting
pred=pd.read_csv(NB04_PRED)
raw_metrics=pd.read_csv(NB04_METRICS)
raw_rank=pd.read_csv(NB04_RANK)

map_cols=df[['master_id','spectral_group_global','full_group_global']].copy()
pred=pred.merge(map_cols,on='master_id',how='left',validate='many_to_one')
assert pred.spectral_group_global.notna().all()
assert pred.full_group_global.notna().all()

def macro_f1_weighted(y_true,y_pred,labels,weights=None):
    return float(f1_score(
        np.asarray(y_true,int),
        np.asarray(y_pred,int),
        labels=labels,
        average='macro',
        zero_division=0,
        sample_weight=weights
    ))

reeval_rows=[]

for (t,modality,model),g in pred[
    pred.modality.isin(['Spectral','TPH_Spectral'])
].groupby(['test_id','modality','model'],sort=True):

    group_col='spectral_group_global' if modality=='Spectral' else 'full_group_global'
    observed=sorted(g.stage_true.astype(int).unique().tolist())

    # Standard row-weighted metric.
    row_macro=macro_f1_weighted(g.stage_true,g.stage_pred,observed)

    # Each feature vector contributes total weight 1.
    freq=g.groupby(group_col)[group_col].transform('size').to_numpy(float)
    w=1.0/freq
    vector_balanced=macro_f1_weighted(g.stage_true,g.stage_pred,observed,w)

    # Collapse repeated vector+label pairs, preserving conflicts as distinct pairs.
    gd=g.drop_duplicates(subset=[group_col,'stage_true']).copy()
    collapsed=macro_f1_weighted(gd.stage_true,gd.stage_pred,observed)

    reeval_rows.append({
        'outer_test':int(t),
        'modality':modality,
        'model':model,
        'n_rows':len(g),
        'n_unique_feature_vectors':int(g[group_col].nunique()),
        'n_unique_feature_label_pairs':len(gd),
        'macro_f1_row_weighted':row_macro,
        'macro_f1_feature_vector_balanced':vector_balanced,
        'macro_f1_feature_label_collapsed':collapsed,
        'delta_vector_balanced_minus_row':vector_balanced-row_macro,
        'delta_collapsed_minus_row':collapsed-row_macro,
    })

reeval=pd.DataFrame(reeval_rows)
reeval.to_csv(TABLE_DIR/'03_nb04_frozen_prediction_duplicate_aware_metrics.csv',index=False)

# Verify standard values reproduce NB04.
verify=reeval.merge(
    raw_metrics[['outer_test','modality','model','macro_f1_observed']],
    on=['outer_test','modality','model'],
    how='left',
    validate='one_to_one'
)
verify['abs_diff']=np.abs(verify.macro_f1_row_weighted-verify.macro_f1_observed)
assert verify.abs_diff.max()<1e-12,verify.sort_values('abs_diff',ascending=False).head()

# Selected models from frozen NB04.
raw_selected_keys=raw_rank[
    raw_rank.modality.isin(['Spectral','TPH_Spectral']) &
    raw_rank.inner_model_rank.eq(1)
][['outer_test','modality','model']].copy()

raw_selected_reeval=raw_selected_keys.merge(
    reeval,on=['outer_test','modality','model'],how='left',validate='one_to_one'
)
assert len(raw_selected_reeval)==8
raw_selected_reeval.to_csv(
    TABLE_DIR/'04_nb04_selected_duplicate_aware_metrics.csv',
    index=False
)

display(raw_selected_reeval.round(4))


In [ ]:
# 5. Build the duplicate-collapsed Spectral dataset

# Keep one row for each experiment × exact spectral vector × observed stage.
# Thus repeated identical labeled rows are collapsed, but label conflicts remain.
dedup=df.drop_duplicates(
    subset=['test_id']+SPECTRAL_FEATURES+['stage'],
    keep='first'
).copy().reset_index(drop=True)

assert len(dedup)==1900

count_compare=pd.DataFrame({
    'test_id':OUTER_TESTS,
    'original_rows':[int((df.test_id==t).sum()) for t in OUTER_TESTS],
    'dedup_vector_stage_rows':[int((dedup.test_id==t).sum()) for t in OUTER_TESTS],
})
count_compare['retained_fraction']=(
    count_compare.dedup_vector_stage_rows/count_compare.original_rows
)
count_compare.to_csv(TABLE_DIR/'05_dedup_sample_size_by_experiment.csv',index=False)

print('Original rows:',len(df))
print('Duplicate-collapsed rows:',len(dedup))
display(count_compare.round(4))


In [ ]:
# 6. Duplicate-aware spectral representations

def snv(X):
    X=np.asarray(X,float)
    mu=X.mean(axis=1,keepdims=True)
    sd=X.std(axis=1,ddof=1,keepdims=True)
    sd=np.where((~np.isfinite(sd)) | (sd<1e-12),1.0,sd)
    return (X-mu)/sd

Xraw=dedup[SPECTRAL_FEATURES].to_numpy(float)
TRANSFORMED={}

TRANSFORMED['RAW_DEDUP']=Xraw.copy()
TRANSFORMED['SNV_DEDUP']=snv(Xraw)

# DOMAIN_Z_ROW_DEDUP: exactly the NB10 domain-statistics definition,
# but applied to the duplicate-collapsed modeling rows.
X_row=np.empty_like(Xraw,dtype=float)

# DOMAIN_Z_UNIQUE_DEDUP: domain means/SDs estimated from unique unlabeled
# spectral vectors only.
X_unique=np.empty_like(Xraw,dtype=float)

domain_stats_rows=[]

for t in OUTER_TESTS:
    idx=np.where(dedup.test_id.to_numpy(int)==t)[0]

    full_t=df[df.test_id.eq(t)]
    fullX=full_t[SPECTRAL_FEATURES].to_numpy(float)

    mu_row=fullX.mean(axis=0,keepdims=True)
    sd_row=fullX.std(axis=0,ddof=0,keepdims=True)
    sd_row=np.where((~np.isfinite(sd_row)) | (sd_row<1e-12),1.0,sd_row)
    X_row[idx]=(Xraw[idx]-mu_row)/sd_row

    unique_t=full_t.drop_duplicates(subset=SPECTRAL_FEATURES)
    uniqueX=unique_t[SPECTRAL_FEATURES].to_numpy(float)
    mu_unique=uniqueX.mean(axis=0,keepdims=True)
    sd_unique=uniqueX.std(axis=0,ddof=0,keepdims=True)
    sd_unique=np.where((~np.isfinite(sd_unique)) | (sd_unique<1e-12),1.0,sd_unique)
    X_unique[idx]=(Xraw[idx]-mu_unique)/sd_unique

    domain_stats_rows.append({
        'test_id':t,
        'n_rows_for_row_stats':len(fullX),
        'n_unique_vectors_for_unique_stats':len(uniqueX),
        'mean_abs_difference_in_band_means':float(np.mean(np.abs(mu_row-mu_unique))),
        'mean_abs_difference_in_band_sds':float(np.mean(np.abs(sd_row-sd_unique))),
    })

TRANSFORMED['DOMAIN_Z_ROW_DEDUP']=X_row
TRANSFORMED['DOMAIN_Z_UNIQUE_DEDUP']=X_unique

domain_stats=pd.DataFrame(domain_stats_rows)
domain_stats.to_csv(TABLE_DIR/'06_domain_z_row_vs_unique_statistics.csv',index=False)

for method,X in TRANSFORMED.items():
    assert X.shape==(len(dedup),18)
    assert np.isfinite(X).all()

display(domain_stats.round(4))


In [ ]:
# 7. Metrics, models and the original compact NB03 grid

def classification_metrics(y_true,y_pred):
    y_true=np.asarray(y_true,int)
    y_pred=np.asarray(y_pred,int)
    observed=sorted(np.unique(y_true).tolist())
    ae=np.abs(y_true-y_pred)
    return {
        'accuracy':float(accuracy_score(y_true,y_pred)),
        'macro_f1_observed':float(f1_score(y_true,y_pred,labels=observed,average='macro',zero_division=0)),
        'macro_f1_fixed_1to5':float(f1_score(y_true,y_pred,labels=ALL_LABELS,average='macro',zero_division=0)),
        'balanced_accuracy':float(balanced_accuracy_score(y_true,y_pred)),
        'mcc':float(matthews_corrcoef(y_true,y_pred)),
        'qwk':float(cohen_kappa_score(y_true,y_pred,labels=ALL_LABELS,weights='quadratic')),
        'ordinal_mae':float(mean_absolute_error(y_true,y_pred)),
        'within_1_stage':float(np.mean(ae<=1)),
        'severe_error_ge2':float(np.mean(ae>=2)),
        'observed_labels':','.join(map(str,observed)),
    }

def fit_predict(model_name,params,X_train,y_train,X_eval,seed):
    y_train=np.asarray(y_train,int)

    if model_name=='OrdinalLogistic':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train); Xq=scaler.transform(X_eval)
        model=mord.LogisticAT(alpha=float(params['alpha']))
        model.fit(Xt,y_train)
        pred=model.predict(Xq).astype(int)

    elif model_name=='PLSDA':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train); Xq=scaler.transform(X_eval)
        classes=np.sort(np.unique(y_train))
        Y=np.zeros((len(y_train),len(classes)),float)
        for j,c in enumerate(classes):
            Y[:,j]=(y_train==c).astype(float)
        ncomp=int(min(params['n_components'],Xt.shape[1],Xt.shape[0]-1))
        model=PLSRegression(n_components=ncomp,scale=False,max_iter=1000)
        model.fit(Xt,Y)
        scores=np.asarray(model.predict(Xq))
        if scores.ndim==1:
            scores=scores[:,None]
        pred=classes[np.argmax(scores,axis=1)].astype(int)

    elif model_name=='SVM_RBF':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train); Xq=scaler.transform(X_eval)
        model=SVC(
            kernel='rbf',C=float(params['C']),gamma=params['gamma'],
            probability=False,cache_size=1000,random_state=seed
        )
        model.fit(Xt,y_train)
        pred=model.predict(Xq).astype(int)

    elif model_name=='RandomForest':
        model=RandomForestClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=params['max_depth'],
            min_samples_leaf=int(params['min_samples_leaf']),
            max_features=params['max_features'],
            random_state=seed,
            n_jobs=N_JOBS
        )
        model.fit(X_train,y_train)
        pred=model.predict(X_eval).astype(int)

    elif model_name=='XGBoost':
        classes=np.sort(np.unique(y_train))
        cmap={c:i for i,c in enumerate(classes)}
        yz=np.array([cmap[c] for c in y_train],int)
        model=xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=len(classes),
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            learning_rate=float(params['learning_rate']),
            subsample=float(params['subsample']),
            colsample_bytree=float(params['colsample_bytree']),
            reg_lambda=1.0,
            min_child_weight=1.0,
            tree_method='hist',
            eval_metric='mlogloss',
            random_state=seed,
            n_jobs=N_JOBS
        )
        model.fit(X_train,yz,verbose=False)
        pred=classes[np.asarray(model.predict(X_eval),int)].astype(int)

    else:
        raise ValueError(model_name)

    return np.clip(pred,1,5).astype(int)

def candidate_grid(model_name):
    if model_name=='OrdinalLogistic':
        return [{'alpha':0.01},{'alpha':0.1},{'alpha':1.0},{'alpha':10.0}]
    if model_name=='PLSDA':
        return [{'n_components':2},{'n_components':4},{'n_components':8}]
    if model_name=='SVM_RBF':
        return [
            {'C':0.5,'gamma':'scale'},
            {'C':2.0,'gamma':'scale'},
            {'C':8.0,'gamma':'scale'},
            {'C':8.0,'gamma':0.03},
        ]
    if model_name=='RandomForest':
        return [
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':18,'min_samples_leaf':1,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':3,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':0.7},
        ]
    if model_name=='XGBoost':
        return [
            {'n_estimators':250,'max_depth':3,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':350,'max_depth':5,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':250,'max_depth':3,'learning_rate':0.10,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':500,'max_depth':2,'learning_rate':0.03,'subsample':0.9,'colsample_bytree':0.9},
        ]
    raise ValueError(model_name)

assert sum(len(candidate_grid(m)) for m in MODELS)==19
print('Model implementations and grid ready.')


## Duplicate-aware nested LOEO

Expected inner fits:

4 duplicate-aware representations × 4 outer folds × 3 inner validation experiments × 19 candidates = **912 fits**.

Because only 1,900 vector–stage rows remain, this is substantially lighter than NB10 despite the same number of fits.


In [ ]:
# 8. Full inner LOEO on duplicate-collapsed Spectral data
inner_rows=[]
leakage_rows=[]
fit_counter=0
inner_start=time.perf_counter()

test_arr=dedup.test_id.to_numpy(int)
y_all=dedup.stage.to_numpy(int)

for method_idx,method in enumerate(METHODS):
    Xall=TRANSFORMED[method]

    for outer_test in OUTER_TESTS:
        dev_tests=[t for t in OUTER_TESTS if t!=outer_test]
        outer_mask=(test_arr==outer_test)

        print(f'\n{method} | outer Test {outer_test} | dev={dev_tests}')

        for model_idx,model_name in enumerate(MODELS):
            candidates=candidate_grid(model_name)

            for cand_idx,params in enumerate(candidates):
                pjson=json.dumps(params,sort_keys=True)

                for inner_val_test in dev_tests:
                    inner_train_tests=[t for t in dev_tests if t!=inner_val_test]
                    tr=np.isin(test_arr,inner_train_tests)
                    va=(test_arr==inner_val_test)

                    assert not np.any(tr & va)
                    assert not np.any(tr & outer_mask)
                    assert not np.any(va & outer_mask)

                    seed=(
                        SEED
                        + method_idx*10000
                        + outer_test*100
                        + inner_val_test*10
                        + cand_idx
                    )

                    t0=time.perf_counter()
                    status='ok'; err=''
                    try:
                        predv=fit_predict(
                            model_name,params,
                            Xall[tr],y_all[tr],
                            Xall[va],seed
                        )
                        met=classification_metrics(y_all[va],predv)
                    except Exception as e:
                        status='error'; err=repr(e)
                        met={k:np.nan for k in [
                            'accuracy','macro_f1_observed','macro_f1_fixed_1to5',
                            'balanced_accuracy','mcc','qwk','ordinal_mae',
                            'within_1_stage','severe_error_ge2'
                        ]}
                        met['observed_labels']=','.join(
                            map(str,sorted(np.unique(y_all[va]).tolist()))
                        )

                    sec=time.perf_counter()-t0
                    fit_counter+=1

                    inner_rows.append({
                        'preprocessing':method,
                        'outer_test':outer_test,
                        'inner_val_test':inner_val_test,
                        'inner_train_tests':','.join(map(str,inner_train_tests)),
                        'model':model_name,
                        'candidate_id':cand_idx,
                        'params_json':pjson,
                        'n_train_dedup':int(tr.sum()),
                        'n_val_dedup':int(va.sum()),
                        'status':status,
                        'error':err,
                        'fit_seconds':sec,
                        **met
                    })

                    leakage_rows.append({
                        'preprocessing':method,
                        'outer_test':outer_test,
                        'inner_val_test':inner_val_test,
                        'model':model_name,
                        'candidate_id':cand_idx,
                        'outer_rows_used_in_train':int(np.sum(tr & outer_mask)),
                        'outer_rows_used_in_validation':int(np.sum(va & outer_mask)),
                        'train_val_overlap_rows':int(np.sum(tr & va)),
                    })

                    if fit_counter%100==0:
                        print(f'  {fit_counter}/912 fits | elapsed {(time.perf_counter()-inner_start)/60:.1f} min')

inner=pd.DataFrame(inner_rows)
leak=pd.DataFrame(leakage_rows)

inner.to_csv(TABLE_DIR/'07_inner_cv_fold_results_duplicate_aware.csv.gz',
             index=False,compression='gzip')
leak.to_csv(TABLE_DIR/'08_inner_cv_leakage_audit.csv',index=False)

assert fit_counter==912
assert inner.status.eq('ok').all(),inner.loc[~inner.status.eq('ok'),['preprocessing','outer_test','model','error']]
assert leak[['outer_rows_used_in_train','outer_rows_used_in_validation','train_val_overlap_rows']].to_numpy().sum()==0

print('\nInner LOEO complete.')
print('Leakage audit: PASS')
print('Minutes:',round((time.perf_counter()-inner_start)/60,2))


In [ ]:
# 9. Candidate aggregation and frozen NB03 lexicographic selection
agg=(inner.groupby(
    ['preprocessing','outer_test','model','candidate_id','params_json'],
    as_index=False
).agg(
    inner_folds=('inner_val_test','nunique'),
    mean_macro_f1_observed=('macro_f1_observed','mean'),
    sd_macro_f1_observed=('macro_f1_observed','std'),
    worst_macro_f1_observed=('macro_f1_observed','min'),
    mean_macro_f1_fixed_1to5=('macro_f1_fixed_1to5','mean'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),
    mean_mcc=('mcc','mean'),
    mean_qwk=('qwk','mean'),
    mean_ordinal_mae=('ordinal_mae','mean'),
    mean_within_1_stage=('within_1_stage','mean'),
    mean_severe_error_ge2=('severe_error_ge2','mean'),
    total_fit_seconds=('fit_seconds','sum')
))

agg['eligible']=agg.inner_folds.eq(3)
agg.to_csv(TABLE_DIR/'09_inner_candidate_summary_duplicate_aware.csv',index=False)

selected_rows=[]
for keys,g in agg[agg.eligible].groupby(['preprocessing','outer_test','model'],sort=True):
    gs=g.sort_values(
        ['mean_macro_f1_observed','worst_macro_f1_observed','mean_qwk','mean_ordinal_mae','candidate_id'],
        ascending=[False,False,False,True,True],
        kind='mergesort'
    )
    selected_rows.append(gs.iloc[0].copy())

selected_hp=pd.DataFrame(selected_rows).reset_index(drop=True)
assert len(selected_hp)==4*4*5

selected_hp['inner_model_rank']=selected_hp.groupby(
    ['preprocessing','outer_test']
).mean_macro_f1_observed.rank(method='min',ascending=False)

selected_hp.to_csv(TABLE_DIR/'10_selected_hyperparameters_duplicate_aware.csv',index=False)
display(selected_hp.head(10))


In [ ]:
# 10. Outer LOEO on duplicate-collapsed Spectral data
outer_rows=[]
outer_fit_counter=0
outer_start=time.perf_counter()

for method_idx,method in enumerate(METHODS):
    Xall=TRANSFORMED[method]

    for outer_test in OUTER_TESTS:
        tr=(test_arr!=outer_test)
        te=(test_arr==outer_test)

        for model_idx,model_name in enumerate(MODELS):
            srow=selected_hp[
                selected_hp.preprocessing.eq(method) &
                selected_hp.outer_test.eq(outer_test) &
                selected_hp.model.eq(model_name)
            ]
            assert len(srow)==1
            s=srow.iloc[0]
            params=json.loads(s.params_json)

            seed=SEED+method_idx*10000+outer_test*100+model_idx*10

            t0=time.perf_counter()
            status='ok'; err=''
            try:
                predv=fit_predict(
                    model_name,params,
                    Xall[tr],y_all[tr],
                    Xall[te],seed
                )
                met=classification_metrics(y_all[te],predv)
            except Exception as e:
                status='error'; err=repr(e)
                met={k:np.nan for k in [
                    'accuracy','macro_f1_observed','macro_f1_fixed_1to5',
                    'balanced_accuracy','mcc','qwk','ordinal_mae',
                    'within_1_stage','severe_error_ge2'
                ]}
                met['observed_labels']=','.join(
                    map(str,sorted(np.unique(y_all[te]).tolist()))
                )

            outer_fit_counter+=1
            outer_rows.append({
                'preprocessing':method,
                'outer_test':outer_test,
                'model':model_name,
                'candidate_id':int(s.candidate_id),
                'params_json':s.params_json,
                'inner_mean_macro_f1':float(s.mean_macro_f1_observed),
                'inner_worst_macro_f1':float(s.worst_macro_f1_observed),
                'inner_model_rank':float(s.inner_model_rank),
                'n_train_dedup':int(tr.sum()),
                'n_test_dedup':int(te.sum()),
                'status':status,
                'error':err,
                'fit_seconds':time.perf_counter()-t0,
                **met
            })

            print(
                f'{method:24s} | T{outer_test} | {model_name:15s} | '
                f'Macro-F1={met["macro_f1_observed"]:.4f}'
            )

outer=pd.DataFrame(outer_rows)
outer.to_csv(TABLE_DIR/'11_outer_loeo_metrics_duplicate_aware.csv',index=False)

assert outer_fit_counter==80
assert outer.status.eq('ok').all(),outer.loc[~outer.status.eq('ok'),['preprocessing','outer_test','model','error']]

print('\nOuter LOEO complete.')
print('Minutes:',round((time.perf_counter()-outer_start)/60,2))


In [ ]:
# 11. Internally selected duplicate-aware model per method and outer test
sel_rows=[]

for keys,g in outer.groupby(['preprocessing','outer_test'],sort=True):
    gs=g.sort_values(
        ['inner_mean_macro_f1','inner_worst_macro_f1','model'],
        ascending=[False,False,True],
        kind='mergesort'
    )
    sel_rows.append(gs.iloc[0].copy())

selected_dedup=pd.DataFrame(sel_rows)[[
    'preprocessing','outer_test','model',
    'inner_mean_macro_f1','macro_f1_observed',
    'macro_f1_fixed_1to5','balanced_accuracy','mcc','qwk','ordinal_mae',
    'n_test_dedup'
]].reset_index(drop=True)

assert len(selected_dedup)==16
selected_dedup.to_csv(TABLE_DIR/'12_selected_model_duplicate_aware.csv',index=False)

# NB10 row-weighted selected Spectral results.
nb10=pd.read_csv(NB10_SELECTED)
nb10_spec=nb10[nb10.modality.eq('Spectral')][[
    'preprocessing','outer_test','model','macro_f1_observed',
    'balanced_accuracy','mcc','qwk','ordinal_mae'
]].copy()

assert set(nb10_spec.preprocessing.unique())=={'RAW','SNV','DOMAIN_Z'}

method_map={
    'RAW_DEDUP':'RAW',
    'SNV_DEDUP':'SNV',
    'DOMAIN_Z_ROW_DEDUP':'DOMAIN_Z'
}

compare_rows=[]
for dedup_method,base_method in method_map.items():
    d=selected_dedup[selected_dedup.preprocessing.eq(dedup_method)].copy()
    b=nb10_spec[nb10_spec.preprocessing.eq(base_method)].copy()

    z=d.merge(
        b,on='outer_test',suffixes=('_dedup','_nb10'),
        validate='one_to_one'
    )

    for r in z.itertuples():
        compare_rows.append({
            'outer_test':int(r.outer_test),
            'dedup_method':dedup_method,
            'nb10_method':base_method,
            'dedup_selected_model':r.model_dedup,
            'nb10_selected_model':r.model_nb10,
            'macro_f1_nb10_row_weighted':float(r.macro_f1_observed_nb10),
            'macro_f1_duplicate_aware':float(r.macro_f1_observed_dedup),
            'delta_duplicate_aware_minus_nb10':float(
                r.macro_f1_observed_dedup-r.macro_f1_observed_nb10
            ),
        })

compare=pd.DataFrame(compare_rows)
compare.to_csv(TABLE_DIR/'13_duplicate_aware_vs_nb10_selected.csv',index=False)

summary=(compare.groupby(['dedup_method','nb10_method'],as_index=False)
         .agg(
             mean_macro_f1_nb10=('macro_f1_nb10_row_weighted','mean'),
             mean_macro_f1_duplicate_aware=('macro_f1_duplicate_aware','mean'),
             mean_delta=('delta_duplicate_aware_minus_nb10','mean'),
             median_delta=('delta_duplicate_aware_minus_nb10','median'),
             min_delta=('delta_duplicate_aware_minus_nb10','min'),
             max_delta=('delta_duplicate_aware_minus_nb10','max'),
             tests_improved=('delta_duplicate_aware_minus_nb10',
                             lambda x:int(np.sum(np.asarray(x)>0)))
         ))
summary.to_csv(TABLE_DIR/'14_duplicate_aware_summary_vs_nb10.csv',index=False)

display(selected_dedup.round(4))
print('\nComparison with NB10 row-weighted:')
display(summary.round(4))


In [ ]:
# 12. Additional comparison: row-stat DOMAIN_Z vs unique-stat DOMAIN_Z after dedup
dz_row=selected_dedup[
    selected_dedup.preprocessing.eq('DOMAIN_Z_ROW_DEDUP')
][['outer_test','model','macro_f1_observed']].rename(columns={
    'model':'model_domain_z_row',
    'macro_f1_observed':'macro_f1_domain_z_row'
})

dz_unique=selected_dedup[
    selected_dedup.preprocessing.eq('DOMAIN_Z_UNIQUE_DEDUP')
][['outer_test','model','macro_f1_observed']].rename(columns={
    'model':'model_domain_z_unique',
    'macro_f1_observed':'macro_f1_domain_z_unique'
})

dz_compare=dz_row.merge(dz_unique,on='outer_test',validate='one_to_one')
dz_compare['delta_unique_stats_minus_row_stats']=(
    dz_compare.macro_f1_domain_z_unique-dz_compare.macro_f1_domain_z_row
)
dz_compare.to_csv(TABLE_DIR/'15_domain_z_unique_vs_row_stats.csv',index=False)

display(dz_compare.round(4))


In [ ]:
# 13. Figures

# A. Duplicate burden
plot_dup=duplicate_audit.pivot(
    index='test_id',
    columns='modality',
    values='feature_vector_duplicate_fraction'
)

fig,ax=plt.subplots(figsize=(8,5))
x=np.arange(len(plot_dup.index))
width=0.36
for j,col in enumerate(plot_dup.columns):
    ax.bar(x+(j-0.5)*width,plot_dup[col].to_numpy(),width=width,label=col)
ax.set_xticks(x,[f'Test {t}' for t in plot_dup.index])
ax.set_ylim(0,1)
ax.set_ylabel('Exact feature-vector duplicate fraction')
ax.set_title('Duplicate burden by experiment and modality')
ax.legend()
ax.grid(axis='y',alpha=0.2)
fig.tight_layout()

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2D_duplicate_burden.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)

# B. NB10 row-weighted vs duplicate-aware Spectral
fig,axes=plt.subplots(1,3,figsize=(14,4.5),sharey=True)

for ax,(base_method,dedup_method) in zip(
    axes,
    [('RAW','RAW_DEDUP'),('SNV','SNV_DEDUP'),('DOMAIN_Z','DOMAIN_Z_ROW_DEDUP')]
):
    b=nb10_spec[nb10_spec.preprocessing.eq(base_method)].sort_values('outer_test')
    d=selected_dedup[selected_dedup.preprocessing.eq(dedup_method)].sort_values('outer_test')

    ax.plot(b.outer_test,b.macro_f1_observed,marker='o',label='NB10 row-weighted')
    ax.plot(d.outer_test,d.macro_f1_observed,marker='s',label='NB11 duplicate-aware')
    ax.set_xticks(OUTER_TESTS)
    ax.set_xlabel('Held-out experiment')
    ax.set_title(base_method)
    ax.grid(alpha=0.2)

axes[0].set_ylabel('Selected-model external Macro-F1')
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',ncol=2,bbox_to_anchor=(0.5,-0.04))
fig.suptitle('Effect of exact spectral duplication on external performance',y=1.02)
fig.tight_layout(rect=[0,0.08,1,1])

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2E_duplicate_aware_vs_nb10.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)

# C. DOMAIN_Z row-stat vs unique-stat
fig,ax=plt.subplots(figsize=(8,5))
z=dz_compare.sort_values('outer_test')
ax.plot(z.outer_test,z.macro_f1_domain_z_row,marker='o',label='DOMAIN_Z using all rows')
ax.plot(z.outer_test,z.macro_f1_domain_z_unique,marker='s',label='DOMAIN_Z using unique vectors')
ax.set_xticks(OUTER_TESTS)
ax.set_xlabel('Held-out experiment')
ax.set_ylabel('Duplicate-aware external Macro-F1')
ax.set_title('Does duplication drive DOMAIN_Z normalization statistics?')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2F_domainz_unique_vs_row_stats.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)


In [ ]:
# 14. Key results and interpretation
key={
    'spectral_duplicate_fraction_by_test':{
        str(int(r.test_id)):float(r.feature_vector_duplicate_fraction)
        for r in duplicate_audit[duplicate_audit.modality.eq('Spectral')].itertuples()
    },
    'tph_spectral_duplicate_fraction_by_test':{
        str(int(r.test_id)):float(r.feature_vector_duplicate_fraction)
        for r in duplicate_audit[duplicate_audit.modality.eq('TPH_Spectral')].itertuples()
    },
    'cross_experiment_shared_spectral_groups':int(len(cross_shared)),
    'rows_in_cross_experiment_shared_groups':int(cross_shared.n_rows.sum()) if len(cross_shared) else 0,
    'cross_experiment_shared_groups_with_label_conflict':int((cross_shared.n_stages>1).sum()) if len(cross_shared) else 0,
    'original_rows':int(len(df)),
    'dedup_vector_stage_rows':int(len(dedup)),
    'selected_duplicate_aware_mean_macro_f1':{
        method:float(
            selected_dedup[selected_dedup.preprocessing.eq(method)].macro_f1_observed.mean()
        )
        for method in METHODS
    },
    'domain_z_unique_minus_row_mean_delta':float(
        dz_compare.delta_unique_stats_minus_row_stats.mean()
    ),
}

(TABLE_DIR/'16_key_results_duplicates.json').write_text(
    json.dumps(key,indent=2,ensure_ascii=False),
    encoding='utf-8'
)

notes=[
    '# exact spectral duplicate sensitivity',
    '',
    '## Duplicate burden',
]
for r in duplicate_audit[duplicate_audit.modality.eq('Spectral')].itertuples():
    notes.append(
        f'- Test {int(r.test_id)}: {r.feature_vector_duplicate_fraction:.1%} exact Spectral-vector duplication; '
        f'{int(r.n_conflicting_feature_vectors)} spectral vectors contain more than one stage label.'
    )

notes += [
    '',
    f'- Exact spectral groups shared across experiments: {len(cross_shared)}.',
    f'- Rows belonging to shared cross-experiment groups: {int(cross_shared.n_rows.sum()) if len(cross_shared) else 0}.',
    f'- Shared groups with stage conflict: {int((cross_shared.n_stages>1).sum()) if len(cross_shared) else 0}.',
    '',
    '## Duplicate-aware external performance',
]

for method in METHODS:
    g=selected_dedup[selected_dedup.preprocessing.eq(method)]
    notes.append(
        f'- {method}: mean selected external Macro-F1 = {g.macro_f1_observed.mean():.4f}.'
    )

notes += [
    '',
    '## Interpretation',
    'The duplicate-collapsed analysis treats every observed spectral-vector/stage association once. '
    'Conflicting labels are preserved rather than resolved by majority vote.',
    '',
    'For Spectral-only modeling, a large difference between NB10 row-weighted results and NB11 '
    'duplicate-aware results indicates that repeated quantized sensor readings materially affect '
    'the apparent validation performance. Conversely, similar results indicate that the experiment-shift '
    'finding is not primarily a consequence of repeated identical spectral vectors.',
    '',
    'TPH+Spectral has almost no exact full-vector duplication, so the severe duplication issue is '
    'specific to the isolated spectral representation rather than the full multisensor observation.',
    '',
    'DOMAIN_Z_UNIQUE_DEDUP uses unlabeled unique target-domain vectors to estimate target normalization '
    'statistics. It remains a transductive diagnostic and must not be described as ordinary single-sample '
    'inductive deployment.'
]

(TEXT_DIR/'duplicate_sensitivity_notes.md').write_text(
    '\n'.join(notes)+'\n',
    encoding='utf-8'
)
print('\n'.join(notes))


In [ ]:
# 15. Manifest, hashes, ZIP and automatic download
versions={
    'python':platform.python_version(),
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'matplotlib':plt.matplotlib.__version__,
    'scikit_learn':sklearn.__version__,
    'mord':getattr(mord,'__version__','0.7/unknown'),
    'xgboost':xgb.__version__,
}

elapsed=time.perf_counter()-start_all

manifest={
    'project':'BANANA_RIPENESS_MULTISENSOR_Q1',
    'notebook':'11_DUPLICATE_AWARE_SPECTRAL_SENSITIVITY',
    'run_id':RUN_ID,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'seed':SEED,
    'source_hashes':source_hashes.to_dict('records'),
    'raw_baseline_run':str(NB04_RUN),
    'normalization_sensitivity_reference_run':str(NB10_RUN),
    'dedup_rule':'one row per experiment + exact 18-band spectral vector + observed stage; conflicts retained',
    'methods':METHODS,
    'models':MODELS,
    'selection_rule':[
        'maximize mean observed-label Macro-F1 across 3 inner experiment folds',
        'maximize worst inner-fold observed-label Macro-F1',
        'maximize mean QWK',
        'minimize mean ordinal MAE'
    ],
    'inner_fits':int(fit_counter),
    'outer_fits':int(outer_fit_counter),
    'outer_label_use_for_preprocessing':False,
    'domain_z_unique_target_use':'unlabeled unique target-domain spectral vectors only',
    'elapsed_minutes':elapsed/60,
    'versions':versions,
    'key_results':key
}

(META_DIR/'run_manifest.json').write_text(
    json.dumps(manifest,indent=2,ensure_ascii=False),
    encoding='utf-8'
)

(META_DIR/'environment.txt').write_text(
    '\n'.join([
        f'platform={platform.platform()}',
        f'cpu_count={os.cpu_count()}',
        *[f'{k}={v}' for k,v in versions.items()]
    ])+'\n',
    encoding='utf-8'
)

readme=[
    '# NB11 — duplicate-aware spectral sensitivity',
    '',
    f'Run ID: `{RUN_ID}`',
    f'Runtime: {elapsed/60:.2f} min',
    '',
    '- Frozen NB04 predictions were reweighted by exact unique feature vectors.',
    '- Spectral training/evaluation was repeated after collapsing exact vector-stage duplicates.',
    '- Stage conflicts were preserved.',
    '- RAW, SNV, DOMAIN_Z row-statistics and DOMAIN_Z unique-vector statistics were compared.',
    '- No external labels were used to fit preprocessing or select models.',
    '',
    'Upload the generated ZIP to ChatGPT for interpretation.'
]
(RUN_DIR/'README_RESULTS.md').write_text(
    '\n'.join(readme)+'\n',
    encoding='utf-8'
)

hash_rows=[]
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file() and p.name!='package_hashes.csv':
        hash_rows.append({
            'relative_path':str(p.relative_to(RUN_DIR)),
            'bytes':p.stat().st_size,
            'sha256':sha256_file(p)
        })
pd.DataFrame(hash_rows).to_csv(
    META_DIR/'package_hashes.csv',
    index=False
)

zip_base=RESULTS_ROOT/f'RESULTS_NB11_DUPLICATE_AWARE_{RUN_ID}'
zip_path=Path(shutil.make_archive(
    str(zip_base),'zip',
    root_dir=RUN_DIR.parent,
    base_dir=RUN_DIR.name
))

latest={
    'notebook':'11_DUPLICATE_AWARE_SPECTRAL_SENSITIVITY',
    'run_id':RUN_ID,
    'run_dir':str(RUN_DIR),
    'zip_path':str(zip_path),
    'zip_bytes':zip_path.stat().st_size,
    'elapsed_minutes':elapsed/60
}
(RESULTS_ROOT/'LATEST_NB11_RUN.json').write_text(
    json.dumps(latest,indent=2),
    encoding='utf-8'
)

print('\n✅ NB11 COMPLETED')
print('Run directory:',RUN_DIR)
print('ZIP:',zip_path)
print('ZIP size (MB):',round(zip_path.stat().st_size/1024**2,2))
print('Elapsed minutes:',round(elapsed/60,2))
print('\nUpload this ZIP to ChatGPT.')

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Automatic browser download did not complete:',repr(e))
    print('ZIP remains safely stored in Drive:',zip_path)
